In [1]:
import json
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import re
from tqdm import tqdm
from typing import Optional

# check if gpu is available
device = 0 if torch.cuda.is_available() else -1
print("Using device:", "GPU" if device == 0 else "CPU")

load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: GPU


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen/Qwen2.5-3B-Instruct',
]

def load_model(model_name: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="cuda" if torch.cuda.is_available() else None,
        torch_dtype="auto",
        trust_remote_code=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # se manca pad token, usiamo eos come pad
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # impostiamo padding_side='left' per decoder-only mode
    tokenizer.padding_side = "left"

    # Creiamo la pipeline (la pipeline userà il tokenizer modificato)
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        dtype=torch.float16 ,#"auto"
        #torch_dtype=torch.float16 #"auto"
    )

    # Assicuriamoci che la pipeline conosca il pad_token_id (alcune versioni lo richiedono esplicitamente)
    pipe.tokenizer.pad_token_id = tokenizer.pad_token_id

    return pipe, model, tokenizer



def destroy_model(model, tokenizer):
    if model is not None:
        del model
    
    if tokenizer is not None:
        del tokenizer

    torch.cuda.empty_cache()

In [3]:
def clean_output(text: str, output_type: str) -> Optional[str]:
    """
    Estrae in modo robusto il contenuto tra <REASONING>...</REASONING> o <ANSWER>...</ANSWER>.
    Gestisce tag mancanti/troncati e differenze di maiuscole/minuscole.
    Restituisce None se non trova nulla.
    """
    if not text:
        return None

    flags = re.DOTALL | re.IGNORECASE

    if output_type == "reasoning":
        # 1) caso ideale: <REASONING>... </REASONING>
        m = re.search(r'<REASONING\s*>(.*?)</REASONING\s*>', text, flags)
        if m:
            return m.group(1).strip()

        # 2) se non c'è chiusura, estrai da <REASONING> fino al tag <ANSWER> (se presente)
        m = re.search(r'<REASONING\s*>(.*?)(?=<ANSWER\b)', text, flags)
        if m:
            return m.group(1).strip()

        # 3) se c'è solo <REASONING> e poi la fine del testo (troncato), prendi tutto il resto
        m = re.search(r'<REASONING\s*>(.*)$', text, flags)
        if m:
            return m.group(1).strip()

        return None

    elif output_type == "answer":
        # 1) caso ideale: <ANSWER>... </ANSWER>
        m = re.search(r'<ANSWER\s*>(.*?)</ANSWER\s*>', text, flags)
        if m:
            answer_text = m.group(1).strip()
            # Cerca solo 0 o 1 nella risposta
            digit_match = re.search(r'[01]', answer_text)
            if digit_match:
                return digit_match.group(0)
            return answer_text

        # 2) se la chiusura manca, cerca una cifra 0/1 subito dopo <ANSWER>
        m = re.search(r'<ANSWER\s*>(\s*[01])\b', text, flags)
        if m:
            return m.group(1).strip()

        # 3) se c'è solo <ANSWER> e poi testo troncato -> prendi il resto e cerca 0/1 dentro
        m = re.search(r'<ANSWER\s*>(.*)$', text, flags)
        if m:
            candidate = m.group(1).strip()
            d = re.search(r'([01])', candidate)
            if d:
                return d.group(1)
            return candidate  # fallback: ritorna quello che c'è

        # 4) fallback ulteriore: se non ci sono tag ma l'output finisce con una riga contenente solo 0 o 1
        m = re.search(r'\n\s*([01])\s*$', text, flags)
        if m:
            return m.group(1)

        return None

    else:
        raise ValueError("Invalid output_type. Must be 'answer' or 'reasoning'.")


def generate_batch_output(pipe, tokenizer, input_text_list, temperature: float = 0.1, batch_size: int = 8):
    """
    input_text_list: lista di frasi
    ritorna: lista di dict con prompt, model_output, reasoning, answer
    """
    prompts = []
    for text in input_text_list:
        prompts.append(f"""        
        Classify whether the use of irony in the response to the user utterance between the tag <USER_UTTERANCE>...</USER_UTTERANCE> is appropriate. Output must be 0 (inappropriate) or 1 (appropriate).

        reason step by step, and then provide your final answer between the tags <ANSWER></ANSWER>.
        Provide your reasoning between the tags <REASONING></REASONING>.

        <USER_UTTERANCE>{text}</USER_UTTERANCE>
        <REASONING>
        """)

    outputs = pipe(
        prompts,
        max_new_tokens=200,
        do_sample=temperature > 0,
        temperature=temperature,
        return_full_text=False,
        batch_size=batch_size,
        pad_token_id=tokenizer.eos_token_id
    )

    results = []
    for prompt, out in zip(prompts, outputs):
        entry = out[0]
        model_output = entry['generated_text'].strip()
        reasoning = clean_output(f"<REASONING>{model_output}", "reasoning")
        answer = clean_output(model_output, "answer")
        results.append((prompt, model_output, reasoning, answer))

    return results



In [4]:
def calculate_aggregated_baseline(dataset_df: pd.DataFrame):
    for model_name in MODELS:
        print(f"Processing model: {model_name}")
        pipe, model, tokenizer = load_model(model_name)
        print("Model and tokenizer loaded.")

        output_file = f'test2_baseline_{model_name.replace("/", "_")}.json'
        output_rows = []

        # --- carica dati già esistenti, se presenti ---
        if os.path.exists(output_file):
            try:
                with open(output_file, 'r', encoding='utf-8') as f:
                    existing_data = json.load(f)
                existing_ids = set([row['unique_id'] for row in existing_data])
                print(f"Found {len(existing_ids)} already processed entries in {output_file}")
                # aggiungi i dati esistenti a output_rows
                output_rows.extend(existing_data)
            except Exception as e:
                print(f"Error loading existing JSON: {e}")
                existing_ids = set()
        else:
            existing_ids = set()

        # filtra dataset_df per saltare già processati
        dataset_df = dataset_df[~dataset_df['unique_id'].isin(existing_ids)]
        print(f"{len(dataset_df)} new entries to process")

        batch_size = 8
        for start in tqdm(range(0, len(dataset_df), batch_size), desc=f"Running {model_name}"):
            batch_df = dataset_df.iloc[start:start+batch_size]
            posts = batch_df['post'].tolist()

            results = generate_batch_output(pipe, tokenizer, posts)

            cleaned_results = []
            for prompt, model_output, reasoning, answer in results:
                invalid_inferences = 0
                while answer not in ['0', '1']:
                    invalid_inferences += 1
                    prompt, model_output, reasoning, answer = generate_batch_output(pipe, tokenizer, [prompt])[0]
                cleaned_results.append((prompt, model_output, reasoning, answer, invalid_inferences))

            for row, (prompt, model_output, reasoning, answer, invalid_inferences) in zip(batch_df.itertuples(), cleaned_results):
                enriched = row._asdict()
                enriched['prompt'] = prompt
                enriched['model_output'] = model_output
                enriched['model_reasoning'] = reasoning
                enriched['model_answer'] = answer
                enriched['invalid_inferences'] = invalid_inferences
                output_rows.append(enriched)

            # salva ad ogni batch
            pd.DataFrame(output_rows).to_json(
                output_file,
                orient='records',
                indent=2,
                force_ascii=False
            )
            print(f"✔ Batch {start//batch_size + 1} salvato → {output_file}")

        destroy_model(model, tokenizer)
        print("Model and tokenizer destroyed.")

In [5]:
# check if exists and open unified corpus in 1_datasets_merge\output\unified_corpus.jsonù
file_path = '../1_datasets_merge/output/unified_corpus.json'
try:
    with open(file_path, 'r', encoding='utf-8') as f:
        unified_corpus = json.load(f)
    print(f"Loaded unified corpus with {len(unified_corpus)} entries")
    df = pd.DataFrame(unified_corpus)
except Exception as e:
    print(f"Error loading unified corpus: {e}")
    raise e

# get only row with language == 'en' and dataset == 'MultiPICo'
df_en_multipico = df[(df['language'] == 'en') & (df['dataset_name'] == 'MultiPICo')]
# show number of entries
print(f"Filtered MultiPICo English entries: {len(df_en_multipico)}")

# get only 700 entries for testing
df_en_multipico = df_en_multipico.head(700)
print(f"Using {len(df_en_multipico)} entries for testing")

calculate_aggregated_baseline(df_en_multipico)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded unified corpus with 25814 entries
Filtered MultiPICo English entries: 2999
Using 700 entries for testing
Processing model: meta-llama/Llama-3.2-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.20s/it]
Device set to use cuda


Model and tokenizer loaded.
Found 700 already processed entries in test2_baseline_meta-llama_Llama-3.2-3B-Instruct.json
0 new entries to process


Running meta-llama/Llama-3.2-3B-Instruct: 0it [00:00, ?it/s]


Model and tokenizer destroyed.
Processing model: Qwen/Qwen2.5-3B-Instruct


Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.62s/it]
Device set to use cuda


Model and tokenizer loaded.
Found 404 already processed entries in test2_baseline_Qwen_Qwen2.5-3B-Instruct.json
0 new entries to process


Running Qwen/Qwen2.5-3B-Instruct: 0it [00:00, ?it/s]


Model and tokenizer destroyed.
